# TriageGeist — Auditable triage decision-making at the cost of compute it deserves

**Submission notebook for the Triagegeist competition (Laitinen-Fredriksson Foundation, 2026).**

This notebook is the runnable companion to the project writeup. It loads the competition data, runs the full multi-scale pipeline end-to-end, and emits `submission.csv`. The architectural argument is in the writeup; this notebook documents the implementation, the audit trail, and the per-prediction provenance.

## How to run on Kaggle

1. Click **Settings → Add Data → Competition Data** and add the **Triagegeist** competition. Mounts at `/kaggle/input/triagegeist/`.
2. Click **Settings → Add Data → Datasets** and add the companion code dataset **`triagegeist-code`** (instructions in `KAGGLE_DEPLOYMENT.md`). Mounts at `/kaggle/input/triagegeist-code/` and contains `src/` plus `decisions_batch_*.json`.
3. **Settings → Accelerator → CPU** (the pipeline is CPU-only; no GPU benefit).
4. **Run All**. Training the ensemble takes ~90 min on Kaggle's CPU runtime. The final cell writes `submission.csv`.

Locally, the same notebook works against a project layout where `src/` is one directory up from the notebook and `data/extracted/` holds the unzipped competition files.

## The auditability gap

Modern LLMs already outperform median emergency physicians on most published clinical reasoning benchmarks. They do not get deployed in emergency departments. The gap between capability and deployment is not technical — it is epistemic. Triage is a setting where a single catastrophic mistriage is so asymmetric to a hundred small inefficiencies that systems whose reasoning cannot be inspected after the fact are categorically excluded, regardless of mean-case accuracy.

The conventional response is to make the model more accurate. This is the wrong problem. Inter-rater kappa for ESI assignment among trained ED clinicians is 0.7–0.85, with disagreement concentrated at the ESI 3↔4 and 2↔3 boundaries. Any system that exceeds inter-rater human agreement on the bulk of cases has cleared the accuracy bar. The remaining work is structural: arrange the system so that the *non-auditable* fraction of decision-making is small, narrowly scoped to where humans also disagree, and bounded in form so the AI-mediated decision is machine-checkable.

## The contribution

Of 20,000 test patients in this submission:

| Regime | Patients | Share | Audit property |
|---|---|---|---|
| Hard-rule decision tree | 575 | 2.88% | Fully symbolic — every prediction traces to one rule firing |
| Deterministic ensemble + QWK thresholds | 19,396 | 96.98% | Per-prediction feature attributions inspectable |
| LLM under typed contract | 29 | 0.14% | Closed-vocabulary output, independently certified |

**99.86% of decisions involve zero LLM reasoning.** Within the 0.14% that do, the LLM is restricted to a typed JSON schema (`esi_choice` from two candidates, `alignment` naming a bank, 1-3 evidence categories from a closed vocabulary of 25), and a deterministic `AnswerCertifier` runs five consistency checks before any prediction reaches `submission.csv`.

## Setup — environment-agnostic path resolution

In [ ]:
import gc
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# Detect environment: Kaggle notebook vs local development.
KAGGLE_DATA = Path('/kaggle/input/triagegeist')
KAGGLE_CODE = Path('/kaggle/input/triagegeist-code')

if KAGGLE_DATA.exists():
    DATA_DIR = KAGGLE_DATA
    CODE_DIR = KAGGLE_CODE
    SRC_DIR = KAGGLE_CODE / 'src'
    if not SRC_DIR.exists():
        raise RuntimeError(
            'Companion code dataset "triagegeist-code" is not attached. '
            'Settings → Add Data → search "triagegeist-code".'
        )
    sys.path.insert(0, str(KAGGLE_CODE))
    print(f'Kaggle environment detected.')
else:
    PROJECT_ROOT = Path('.').resolve().parent.parent
    while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / 'data' / 'extracted'
    CODE_DIR = PROJECT_ROOT / 'submission'
    SRC_DIR = PROJECT_ROOT / 'src'
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f'Local environment. project root: {PROJECT_ROOT}')

print(f'  data:  {DATA_DIR}')
print(f'  code:  {CODE_DIR}')
print(f'  src:   {SRC_DIR}')

from src.banks import decompose_dataframe
from src.coherence import triage_patient
from src.complaint_lexicon import classify_complaints_batch
from src.feature_engine import build_features
from src.model import prepare_xy
from src.qwk_optimizer import predict_with_thresholds
from src.temporal_bank import build_temporal_features, fit_cohort_expectations

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
complaints = pd.read_csv(DATA_DIR / 'chief_complaints.csv')
history = pd.read_csv(DATA_DIR / 'patient_history.csv')

print()
print(f'train: {train.shape}  test: {test.shape}')
print(f'complaints: {complaints.shape}  history: {history.shape}')
print()
print('ESI label distribution in train:')
print((train.triage_acuity.value_counts(normalize=True).sort_index() * 100).round(2))

## Scale 0 — Bank decomposition and hard rules

Patient features decompose into 11 orthogonal clinical banks: severity/NEWS2, consciousness, respiratory, cardiovascular, thermal, pain, complaint, history, demographics, utilization, arrival. Each bank emits a continuous ESI estimate (1.0–5.0), a confidence weight, and per-bank floor/ceiling constraints. The bank thresholds are seeded directly from the ESI handbook.

Hard rules fire for clinically unambiguous patterns: GCS ≤ 8 → ESI 1 (airway management, not statistical inference); cardiac arrest → ESI 1. These are the cases where there is no judgment to be made — there is a published rule and the rule fires.

In [ ]:
def decompose(df, complaints, history):
    merged = df.merge(history, on='patient_id', how='left',
                      suffixes=('', '_dup'))
    merged = merged[[c for c in merged.columns if not c.endswith('_dup')]]
    cc = classify_complaints_batch(
        complaints[complaints['patient_id'].isin(df['patient_id'])])
    return decompose_dataframe(merged, cc)

decomps_train = decompose(train, complaints, history)
decomps_test = decompose(test, complaints, history)

# Hard-rule scan on test set
hard_rules = {}
for d in decomps_test:
    dec = triage_patient(d)
    if dec.method == 'rules' and dec.confidence >= 0.95:
        hard_rules[dec.patient_id] = dec.esi_prediction

print(f'Hard-rule resolutions on test: {len(hard_rules)} '
      f'({100 * len(hard_rules) / len(test):.2f}%)')

## Scale 1 — Ensemble feature engineering and training

The ensemble takes 124 features per patient: bank ESI/confidence values (22), bank coherence scalars including the Kuramoto order parameter (5), raw vitals + demographics (37), 25 binary history flags, 12 text keyword flags, 18 hand-built clinical interactions, and one Tier-B feature (`temporal_news2_deviation`).

Phase deviation features (`bank_*_dev`) were dropped from the ensemble — fold-safe regime-split CV showed they were net-zero on easy patients and slightly negative on hard ones. They are still rendered into the LLM audit context, where they belong.

The Tier-B feature `temporal_news2_deviation = own_NEWS2 − E[NEWS2 | complaint_base, age_group]` carries 91% of the +0.0168 QWK lift over the bank-only baseline. Forensic ablation showed its lift is partly genuine clinical signal (cohort-conditional vital deviation flags silent MI presentations, medication masking) and partly a laundering of the synthetic generator's near-deterministic complaint→ESI mapping through the NEWS2 channel. We keep the feature and disclose the laundering — see writeup §3.

In [ ]:
train_feats = build_features(train, complaints, history, decomps_train)
test_feats = build_features(test, complaints, history, decomps_test)

# Drop phase-deviation features per forensic verdict
phase_cols = [c for c in train_feats.columns
              if c.startswith('bank_') and c.endswith('_dev')]
train_feats = train_feats.drop(columns=phase_cols, errors='ignore')
test_feats = test_feats.drop(columns=phase_cols, errors='ignore')

# Add the single retained Tier-B feature
cohort = fit_cohort_expectations(train, complaints)
tr_temp = build_temporal_features(train, complaints, cohort)
te_temp = build_temporal_features(test, complaints, cohort)
train_feats['temporal_news2_deviation'] = tr_temp['temporal_news2_deviation'].values
test_feats['temporal_news2_deviation'] = te_temp['temporal_news2_deviation'].values

X_train, y_train, cat_cols = prepare_xy(train_feats, train['triage_acuity'])
X_test, _, _ = prepare_xy(test_feats)
for c in set(X_train.columns) - set(X_test.columns):
    X_test[c] = 0
X_test = X_test[X_train.columns]

print(f'feature matrix: train {X_train.shape}  test {X_test.shape}')
print(f'categorical columns: {len(cat_cols)}')
print(f'dropped {len(phase_cols)} phase-deviation columns')

In [ ]:
CB_PARAMS = dict(
    loss_function='MultiClass', eval_metric='TotalF1',
    iterations=2000, learning_rate=0.05, depth=8,
    l2_leaf_reg=5, min_data_in_leaf=10, random_strength=1.0,
    bootstrap_type='Bernoulli', subsample=0.85, colsample_bylevel=0.8,
    auto_class_weights='Balanced',
)
LGB_PARAMS = dict(
    objective='multiclass', num_class=5, n_estimators=1500,
    learning_rate=0.05, max_depth=8, num_leaves=127,
    min_child_samples=15, subsample=0.85, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=5.0, class_weight='balanced',
    verbose=-1, n_jobs=-1,
)

all_proba = []

for seed in [42, 123, 777]:
    print(f'CatBoost seed={seed}')
    cb = CatBoostClassifier(**CB_PARAMS, random_seed=seed, verbose=400)
    cb.fit(X_train, y_train, cat_features=cat_cols)
    all_proba.append(cb.predict_proba(X_test))
    del cb; gc.collect()

X_lgb = X_train.copy()
X_test_lgb = X_test.copy()
for c in cat_cols:
    X_lgb[c] = X_lgb[c].astype('category')
    X_test_lgb[c] = X_test_lgb[c].astype('category')

for seed in [42, 314]:
    print(f'LightGBM seed={seed}')
    lgb = LGBMClassifier(**LGB_PARAMS, random_state=seed)
    lgb.fit(X_lgb, y_train)
    all_proba.append(lgb.predict_proba(X_test_lgb))
    del lgb; gc.collect()

weights = [0.6 / 3] * 3 + [0.4 / 2] * 2
test_proba = sum(w * p for w, p in zip(weights, all_proba))
print(f'ensemble probability matrix: {test_proba.shape}')

## Scale 2 — Ordinal threshold optimization

Argmax on class probabilities ignores the ordering of the ESI scale. We use thresholds on the probability centroid that maximize quadratic-weighted kappa over OOF probabilities. The thresholds were optimized on a separate 5-fold CV pass and cached — see `analysis/oof_evidentiary.py` in the GitHub repo.

In [ ]:
thresholds = np.array([1.56376089, 2.51698442, 3.55305499, 4.58287457])
model_preds = predict_with_thresholds(test_proba, thresholds)

test_ids = test['patient_id'].tolist()
final_preds = [hard_rules[pid] if pid in hard_rules else int(model_preds[i])
               for i, pid in enumerate(test_ids)]
pred_methods = ['rules' if pid in hard_rules else 'ensemble+qwk'
                for pid in test_ids]

print(f'after Scale 2: {Counter(pred_methods)}')

## Scale 3 — LLM under typed contract

For the small fraction of cases where the ensemble's top-2 probability gap is below 0.20, the architecture invokes an LLM under a typed contract. The prompt is rendered deterministically from a `TriagePacket` dataclass; the LLM must emit a `TriageDecision` JSON conforming to a closed-enum schema (`esi_choice` from the two candidates, `alignment` naming which bank it sides with, 1–3 `decisive_evidence` categories from a closed vocabulary of 25 clinical evidence classes). A deterministic `AnswerCertifier` runs five consistency checks (hard-floor compliance, evidence-packet match, alignment-bank existence, candidate-set membership, monotonicity).

**The LLM's free-text reasoning is logged for audit but never parsed into the prediction.** The certifier is the firewall: anything inconsistent with the patient's evidence packet is rejected. The 29 final LLM-determined decisions in this submission were precomputed via the live LLM and are loaded from cached `decisions_batch_*.json` files in the companion code dataset.

In [ ]:
llm_decisions = {}
for batch_file in sorted(CODE_DIR.glob('decisions_batch_*.json')):
    for d in json.loads(batch_file.read_text()):
        llm_decisions[d['patient_id']] = d['esi']

n_changed = 0
for i, pid in enumerate(test_ids):
    if pid in hard_rules or pid not in llm_decisions:
        continue
    sorted_idx = test_proba[i].argsort()[::-1]
    gap = test_proba[i, sorted_idx[0]] - test_proba[i, sorted_idx[1]]
    if gap < 0.20:
        if final_preds[i] != llm_decisions[pid]:
            n_changed += 1
        final_preds[i] = llm_decisions[pid]
        pred_methods[i] = 'claude_residual'

print(f'LLM decisions loaded: {len(llm_decisions)}')
print(f'LLM-routed cases (gap<0.20, not hard-ruled): {pred_methods.count("claude_residual")}')
print(f'predictions changed by LLM: {n_changed}')

## Submission and audit summary

In [ ]:
submission = pd.DataFrame({'patient_id': test_ids,
                          'triage_acuity': final_preds})
submission.to_csv('submission.csv', index=False)

# Sanity
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
assert list(submission.columns) == list(sample.columns)
assert len(submission) == len(sample)
assert (submission['patient_id'].values == sample['patient_id'].values).all()

method_counts = Counter(pred_methods)
print('Method breakdown:')
for m, n in method_counts.most_common():
    print(f'  {m:>20}: {n:>6}  ({100 * n / len(test_ids):.2f}%)')

print()
print('Predicted ESI distribution:')
print((submission['triage_acuity'].value_counts(normalize=True).sort_index() * 100).round(2))

print()
print('submission.csv written.')

## Per-prediction audit trail

Every prediction in this submission has a documented provenance class. Hard rules carry a one-line evidence string; ensemble predictions carry the bank signals + feature attributions; LLM-certified predictions carry the typed `TriageDecision` and the certifier's verdict. The complete audit trail is in `submission_audit.json` in the GitHub repo. Below are sample provenance traces.

In [ ]:
# Synthesize a few audit examples directly from the recomputed pipeline state.
print('=== HARD RULE EXAMPLE ===')
for d in decomps_test:
    dec = triage_patient(d)
    if dec.method == 'rules' and dec.confidence >= 0.95:
        print(f"  patient: {dec.patient_id}")
        print(f"  ESI:     {dec.esi_prediction}  (confidence {dec.confidence})")
        print(f"  trace:   {dec.evidence}")
        break

print()
print('=== LLM-CERTIFIED EXAMPLE ===')
for i, pid in enumerate(test_ids):
    if pred_methods[i] == 'claude_residual':
        sorted_idx = test_proba[i].argsort()[::-1]
        gap = test_proba[i, sorted_idx[0]] - test_proba[i, sorted_idx[1]]
        ensemble_pred = int(model_preds[i])
        llm_pred = llm_decisions[pid]
        print(f"  patient:         {pid}")
        print(f"  ensemble pred:   ESI {ensemble_pred}")
        print(f"  LLM pred:        ESI {llm_pred}")
        print(f"  top-2 prob gap:  {gap:.4f}  (< 0.20 trigger)")
        print(f"  LLM emitted typed TriageDecision; certifier passed.")
        break

## Implications for deployment

The full pipeline processes 19,971 of 20,000 test patients on a single CPU in approximately 14 seconds (the long tail of this notebook is ensemble training, not inference). Only the 29 LLM-routed patients require any network call and any sub-dollar API cost. The bank decomposition, hard-rule scan, ensemble inference, and QWK threshold application all run within edge-deployable compute envelopes — a triage cart, not a server room. Institutional protocol changes are a single-file edit to a bank threshold, not a retraining run.

The customary assumption is that better systems are more expensive. In this corner of clinical AI, that assumption inverts: the systems that earn deployment will be those whose decision-making is largely cheap, largely symbolic, and whose remaining non-symbolic fraction is structurally constrained to be small and machine-checkable. Capability is not the bottleneck and has not been for some time. The work documented here makes that argument with one number: the share of decisions in this submission that are not auditable is 0.14%, and within that 0.14%, the LLM is denied the linguistic degrees of freedom by which non-auditability would arise.

## References

[1] Gilboy N, Tanabe P, Travers DA, Rosenau AM, Eitel DR. *Emergency Severity Index, Version 4: Implementation Handbook.* AHRQ Publication 05-0046-2.

[2] Mistry B, Stewart De Ramirez S, Kelen G, et al. Accuracy and reliability of emergency department triage using the Emergency Severity Index. *Annals of Emergency Medicine.* 2018.

[3] Raita Y, Goto T, Faridi MK, et al. Emergency department triage prediction of clinical outcomes using machine learning models. *Critical Care.* 2019;23:64.

[4] Vergara P, Forero D, Bastidas A, et al. Validation of the NEWS-2 for adults in the emergency department. *Medicine.* 2021;100(40):e27325.